# SatQuery AI: train the remote-sensing vision-language model on Kaggle

This notebook adapts **Qwen3-VL-2B** to satellite imagery with **BigEarthNet.txt** (Sentinel-2 optical +
Sentinel-1 radar, with yes/no, multiple-choice, caption and box questions), then scores it against the
untrained model on the benchmark split.

**Before you run it (one time):**
1. Right panel → **Session options** → **Accelerator: GPU T4 x2** (or P100).
2. Same panel → **Internet: On**. (Kaggle asks for phone verification the first time.)
3. Top right → **Save Version** → **Save & Run All (Commit)** → Save.
   This runs in the background for up to 12 hours; you can close the tab.

**When it finishes:** open the version → **Output** tab → download `satquery_vlm_results.zip`
(the trained adapter, ~70 MB, plus scores). Unzip it into the SatQuery folder; see the last cell.

Every phase has a hard time budget, so the run always finishes inside Kaggle's 12-hour limit and saves its
results: the imagery download stops after at most 5 hours (training then uses the images that arrived),
training gets the time left minus 2 hours, and scoring stops on time with the examples it has done.
The trained adapter is zipped as soon as training ends, so it is saved even if scoring is cut short.

In [ ]:
import os, time, json, random, subprocess, torch
START = time.time()
SESSION_HOURS = 11.5          # Kaggle stops a run at 12 h; keep a margin
assert torch.cuda.is_available(), "No GPU: right panel → Session options → Accelerator → GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")

def minutes_left():
    return (START + SESSION_HOURS * 3600 - time.time()) / 60

## Settings
Defaults train on about 12,000 examples (3,000 per question type) and score 200 benchmark examples per type.

In [ ]:
PER_TYPE = 3000          # training examples per type: yes/no, multiple choice, caption, box
BENCH_PER_TYPE = 200     # benchmark examples per type to score
DOWNLOAD_MAX_MIN = 300   # stop the imagery download after 5 h and train on what arrived
EVAL_RESERVE_MIN = 120   # minutes kept after training for scoring and packaging
DATA = "/tmp/bentxt"     # scratch disk (not saved)
OUT = "/kaggle/working"  # saved as the notebook's output
REPO = "/tmp/SatQuery"
ARGS = f"--out {DATA} --per-type {PER_TYPE} --bench-per-type {BENCH_PER_TYPE}"

## 1. Get the code and libraries (about 2 minutes)

In [ ]:
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/Vansh-Vaid/SatQuery.git", REPO], check=True)
os.chdir(REPO)
!pip install -q "transformers>=4.57" peft accelerate zstandard rasterio opencv-python-headless reportlab python-multipart
# Kaggle ships torchao 0.10; current peft refuses any torchao older than 0.16 and crashes when LoRA is attached.
# SatQuery does not use torchao, so remove it.
!pip uninstall -y -q torchao
# One-minute check that LoRA training, saving, loading and generation all work, BEFORE the hours-long download.
# A tiny random model is used, so this tests the code path, not accuracy. A failure stops the notebook here.
subprocess.run(["python", "scripts/smoke_vlm_pipeline.py"], check=True)
!git log --oneline -1

## 2. Download only the images the subset needs (at most 5 hours)
BigEarthNet's imagery is 118 GB, more than Kaggle's disk. This step streams the archives from Zenodo once,
keeps only the selected patches, and saves them as small PNGs rendered exactly as the app renders them.
If the connection drops it retries; finished patches are kept.
Zenodo is sometimes slow. If the download hits its time limit, it stops and keeps what it has: examples
without an optical image are dropped, and examples without radar become single-image examples.

In [ ]:
dl_budget = int(min(DOWNLOAD_MAX_MIN, minutes_left() - EVAL_RESERVE_MIN - 90))
print(f"downloading for at most {dl_budget} minutes")
!python training/fetch_bigearthnet_subset.py {ARGS} --time-limit-min {dl_budget}
print(f"{minutes_left():.0f} minutes of session left")

## 3. Build the training files

In [ ]:
!python training/prepare_bigearthnet_txt.py --rendered {ARGS}
!wc -l {DATA}/*.jsonl

## 4. Check the box coordinate order
BigEarthNet.txt writes boxes as `[a b, c d]`. The app assumes (x, y). Red is the (x, y) reading and blue is
(row, column). The box should cover the named land cover; if blue fits better, tell Claude and it will
switch the order (training is consistent either way).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
rows = [json.loads(l) for l in open(f"{DATA}/train.jsonl") if '"bounding box"' in l][:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.flat, rows):
    im = Image.open(os.path.join(DATA, r["images"][0])); w, h = im.size
    a, b, c, d = [v / 1000 for v in json.loads(r["answer"])[0]["bbox_2d"]]
    ax.imshow(im)
    ax.add_patch(plt.Rectangle((a * w, b * h), (c - a) * w, (d - b) * h, fill=False, ec="red", lw=2))
    ax.add_patch(plt.Rectangle((b * w, a * h), (d - b) * w, (c - a) * h, fill=False, ec="blue", lw=2, ls="--"))
    ax.set_title(r["category"][:40], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/box_order_check.png", dpi=80); plt.show()

## 5. Train (LoRA, one epoch or until the time budget runs out)

In [ ]:
import zipfile

def package():
    """Zip everything in the output folder; called after training and again at the end."""
    with zipfile.ZipFile(f"{OUT}/satquery_vlm_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(OUT):
            for f in files:
                if not f.endswith(".zip"):
                    full = os.path.join(root, f)
                    z.write(full, os.path.relpath(full, OUT))

budget = int(minutes_left() - EVAL_RESERVE_MIN)
assert budget > 20, f"only {budget} minutes left for training; lower DOWNLOAD_MAX_MIN and run again"
print(f"training for at most {budget} minutes")
train = subprocess.run(["python", "training/finetune_vlm.py", "--data", DATA, "--out", f"{OUT}/satquery-vlm-lora",
                        "--epochs", "1", "--eval-every", "100", "--time-limit-min", str(budget)])
package()   # zip whatever exists (adapter and training log), even if scoring is cut short
assert train.returncode == 0, "training failed: see the error above (the zip still has the training log)"
assert os.path.exists(f"{OUT}/satquery-vlm-lora/adapter_config.json"), "training finished but saved no adapter"
print(f"adapter saved; {minutes_left():.0f} minutes left for scoring")

## 6. Score it: trained vs untrained model on the benchmark split
The benchmark split of BigEarthNet.txt was manually verified by its authors and never used for training.

In [ ]:
!python eval/convert.py bentxt --root {DATA} --split bench --out {DATA}/bench_eval.jsonl
rows = [json.loads(l) for l in open(f"{DATA}/bench_eval.jsonl")]
random.Random(0).shuffle(rows)            # mix the question types so a partial run is still representative
with open(f"{DATA}/bench_mix.jsonl", "w") as f:
    f.writelines(json.dumps(r) + "\n" for r in rows)
print(len(rows), "benchmark examples")

os.environ["SATQUERY_VLM_BACKEND"] = "hf"
os.environ["SATQUERY_VLM_ADAPTER"] = f"{OUT}/satquery-vlm-lora"
lora_min = int(max(10, (minutes_left() - 15) * 0.6))
print(f"scoring the trained model for at most {lora_min} minutes")
lora_eval = subprocess.run(["python", "eval/run_eval.py", "--data", f"{DATA}/bench_mix.jsonl", "--mode", "vlm",
                            "--out", f"{OUT}/results/bench_lora", "--time-limit-min", str(lora_min)])
if lora_eval.returncode != 0:
    print("WARNING: scoring the trained model failed; its numbers are not valid (see the error above)")

In [ ]:
# the untrained model on the same examples, for comparison, in the time that is left
base_min = int(minutes_left() - 15)
os.environ.pop("SATQUERY_VLM_ADAPTER", None)
if base_min >= 5:
    print(f"scoring the untrained model for at most {base_min} minutes")
    !python eval/run_eval.py --data {DATA}/bench_mix.jsonl --mode vlm --out {OUT}/results/bench_base --time-limit-min {base_min}
else:
    print("no time left to score the untrained model; skipping the comparison")

## 7. Package the results

In [ ]:
for k in ("bench_lora", "bench_base"):
    p = f"{OUT}/results/{k}/metrics.json"
    if os.path.exists(p):
        print(f"--- {k} ---"); print(open(p).read())
package()
print("Download satquery_vlm_results.zip from the Output tab.")
print(f"Total run time: {(time.time() - START) / 3600:.1f} h")

## Using the trained model in the app
1. Unzip `satquery_vlm_results.zip` into the SatQuery folder, so you have `satquery-vlm-lora/` and `results/`.
2. Move `satquery-vlm-lora` into `checkpoints/`.
3. The model needs a GPU to answer quickly. On a laptop without one, keep it on Kaggle and serve it with
   `scripts/serve_vlm.py`, or just share the `results/` folder with Claude to update the README numbers.